# Spark Preparation
We check if we are in Google Colab.  If this is the case, install all necessary packages.

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 3.3.2 with hadoop 3.3, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab.
Learn more from [A Must-Read Guide on How to Work with PySpark on Google Colab for Data Scientists!](https://www.analyticsvidhya.com/blog/2020/11/a-must-read-guide-on-how-to-work-with-pyspark-on-google-colab-for-data-scientists/)

In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
    !apt-get install openjdk-8-jdk-headless -qq > /dev/null
    !wget -q https://dlcdn.apache.org/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
    !tar xf spark-3.3.2-bin-hadoop3.tgz
    !mv spark-3.3.2-bin-hadoop3 spark
    !pip install -q findspark
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["SPARK_HOME"] = "/content/spark"

# Start a Local Cluster

In [3]:
from functools import reduce
from pyspark.sql.functions import (col, trim, lower, regexp_replace, sum, udf, to_timestamp,split, datediff, substring, length,
    current_timestamp, when, datediff, try_to_timestamp, to_date)
from pythainlp import word_tokenize
from pyspark.sql.types import ArrayType, StringType
from pythainlp.corpus import thai_stopwords



In [4]:
spark_url = 'local'
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder \
    .appName("TraffyFondueDataCleaning") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [5]:
sc = spark.sparkContext

file_path = '../data/bangkok_traffy.csv'

## schema

In [6]:
traffy_schema = StructType([
    # ตัวระบุเฉพาะ
    StructField("ticket_id", StringType(), True),
    
    # ข้อมูลปัญหาและการจัดการ
    StructField("type", StringType(), True),         # หมวดหมู่ปัญหา
    StructField("organization", StringType(), True), # หน่วยงานที่รับผิดชอบ
    StructField("comment", StringType(), True),      # ข้อความร้องเรียน (สำคัญสำหรับ LLM)
    StructField("photo", StringType(), True),
    StructField("photo_after", StringType(), True),
    
    # ข้อมูลพิกัดและตำแหน่ง
    StructField("coords", StringType(), True),       # พิกัด Lat/Long (เก็บเป็น String ก่อนแล้วค่อย Parse)
    StructField("address", StringType(), True),
    StructField("subdistrict", StringType(), True),
    StructField("district", StringType(), True),
    StructField("province", StringType(), True),
    
    # ข้อมูลเวลาและสถานะ
    StructField("timestamp", StringType(), True),    # วันที่สร้าง (เก็บเป็น String ก่อนแล้วค่อย Cast เป็น Timestamp)
    StructField("state", StringType(), True),        # สถานะปัจจุบัน (ใช้ในการ Filter Active Issues)
    
    # ข้อมูลการตอบรับและกิจกรรม
    StructField("star", FloatType(), True),          # เรทติ้ง 0-5
    StructField("count_reopen", IntegerType(), True), # จำนวนครั้งที่เปิดซ้ำ
    StructField("last_activity", StringType(), True)  # วันที่กิจกรรมล่าสุด (เก็บเป็น String ก่อน)
])

In [7]:
df_traffy = spark.read.csv(
    file_path,
    header=True,
    schema=traffy_schema,
    multiLine=True, # สำคัญ: หาก 'comment' หรือ 'address' มีหลายบรรทัด
    escape='"' # สำคัญ: หากมีเครื่องหมายคำพูดในข้อความ
)

In [ ]:
# df_traffy.select("last_activity").show(10, False)


+-----------------------------+
|last_activity                |
+-----------------------------+
|2022-06-04 15:34:14.609206+00|
|2022-06-21 08:21:09.532782+00|
|2022-06-06 01:17:12.272904+00|
|2022-09-08 08:35:43.784519+00|
|2022-08-12 07:18:44.884945+00|
|2023-03-14 12:09:14.947437+00|
|2023-05-17 06:11:32.463984+00|
|2024-11-26 04:17:39.760344+00|
|2022-06-24 06:32:34.671236+00|
|2022-06-20 13:12:04.99444+00 |
+-----------------------------+
only showing top 10 rows


In [ ]:
# print(f"จำนวนแถวเริ่มต้น: {df_traffy.count()}")
# df_traffy.printSchema()

จำนวนแถวเริ่มต้น: 787026
root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- photo: string (nullable = true)
 |-- photo_after: string (nullable = true)
 |-- coords: string (nullable = true)
 |-- address: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- state: string (nullable = true)
 |-- star: float (nullable = true)
 |-- count_reopen: integer (nullable = true)
 |-- last_activity: string (nullable = true)



In [10]:


# ลิสต์หมวดหมู่หลักที่ส่งผลต่อมูลค่าอสังหาฯ และความน่าอยู่
livability_types = [
    "ถนน",
    "ทางเท้า",
    "ความปลอดภัย",
    "แสงสว่าง",
    "ความสะอาด",
    "กีดขวาง",
    "ท่อระบายน้ำ",
    "น้ำท่วม",
    "ต้นไม้",
    "PM2",
    "จราจร",
    "สะพาน"
]

# กรองข้อมูลตาม 'type' (หมวดหมู่ปัญหา)
# ใช้วิธี 'isin' ที่ตรงไปตรงมาที่สุด
df_filtered_type = df_traffy.filter(
    reduce(lambda a, b: a | b, [col("type").contains(t) for t in livability_types])
)






In [11]:
df_final_spatial = df_filtered_type.withColumn(
    "lon_raw", 
    trim(split(col("coords"), ",").getItem(0)).cast("float")
).withColumn(
    "lat_raw", 
    trim(split(col("coords"), ",").getItem(1)).cast("float")
).withColumn(
    # Set the corrected columns
    "lon", col("lon_raw")
).withColumn(
    "lat", col("lat_raw")
).filter(
    # Filter for valid Thai coordinates (Roughly: lat between 5-21, lon between 97-105)
    (col("lat") >= 5) & (col("lat") <= 21) & 
    (col("lon") >= 97) & (col("lon") <= 105)
)


print(f"DataFrame สุดท้ายพร้อมสำหรับ LLM/Join: {df_final_spatial.count()} แถว")
df_final_spatial.select("lat", "lon").show(5, truncate=False)


DataFrame สุดท้ายพร้อมสำหรับ LLM/Join: 596369 แถว
+--------+---------+
|lat     |lon      |
+--------+---------+
|13.81865|100.53084|
|13.67891|100.66709|
|13.7206 |100.52649|
|13.81853|100.53099|
|13.8228 |100.59165|
+--------+---------+
only showing top 5 rows


In [12]:
from pyspark.sql.functions import floor, count, round
df_grid_prep = df_final_spatial.withColumn(
    "grid_lat",
    # (floor(lat * 100) / 100) จะตัดทศนิยมให้เหลือ 2 ตำแหน่ง (0.01, 0.02, ...)
    floor(col("lat") * 100) / 100
).withColumn(
    "grid_lon",
    floor(col("lon") * 100) / 100
)

# --- 2. จัดกลุ่มและนับจำนวน ---
# นับจำนวน Ticket ในแต่ละกริด
df_coords_grouped = df_grid_prep.groupBy(
    "grid_lat", 
    "grid_lon"
).agg(
    count("ticket_id").alias("incident_count")
).orderBy(col("incident_count").asc())

# แสดง 20 พื้นที่ที่มีปัญหาหนาแน่นที่สุด (Top 20 Grid)
# print("Top 20 Grid Bins (0.01 deg) by Incident Count:")
# df_coords_grouped.show(20)

# --- 3. ตรวจสอบความสะอาดของข้อมูล (Validation) ---
# ตรวจสอบช่วงค่า Min/Max ของกริดที่สร้างขึ้น
df_coords_grouped.describe().show()

+-------+------------------+------------------+----------------+
|summary|          grid_lat|          grid_lon|  incident_count|
+-------+------------------+------------------+----------------+
|  count|              1606|              1606|            1606|
|   mean|13.777254047322531|100.60399128268968|368.252801992528|
| stddev|0.6132297371276336|0.2753475468840265|613.909670647796|
|    min|              6.95|             97.93|               0|
|    max|              19.9|            102.81|            5234|
+-------+------------------+------------------+----------------+



In [13]:
df_trim_string = df_final_spatial.withColumn(
    "timestamp_str", 
    substring(col("timestamp"), 1, 19) # เริ่มจาก index 1, เอา 19 ตัวอักษร
).withColumn(
    "last_activity_str", 
    substring(col("last_activity"), 1, 19) # ทำเหมือนกันกับ last_activity
)

# Format ที่ใช้หลังตัด:
TIMESTAMP_FORMAT_SIMPLE = "yyyy-MM-dd HH:mm:ss"

# 2. แปลง String เป็น Timestamp (TimestampType)
df_time_prep = df_trim_string.withColumn(
    "timestamp_dt", 
    to_timestamp(col("timestamp_str"), TIMESTAMP_FORMAT_SIMPLE)
).withColumn(
    "last_activity_dt", 
    to_timestamp(col("last_activity_str"), TIMESTAMP_FORMAT_SIMPLE)
)

# กรองแถวที่แปลง timestamp ไม่ได้ (Timestamp/last_activity เป็น NULL หลังแปลง)
df_time_prep = df_time_prep.filter(
    col("timestamp_dt").isNotNull() 
)

# 3. แปลงเป็น Date และคำนวณ DaysToFix
df_time_prep = df_time_prep.withColumn("timestamp_date", to_date(col("timestamp_dt")))
df_time_prep = df_time_prep.withColumn("last_activity_date", to_date(col("last_activity_dt")))

df_final_ready = df_time_prep.withColumn(
    "DaysToFix",
    when(
        # ถ้า state = 'เสร็จสิ้น'
        col("state") == "เสร็จสิ้น",
        datediff(col("last_activity_date"), col("timestamp_date"))
    ).otherwise(
        # ถ้า state = 'กำลังดำเนินการ' หรือ 'รอรับเรื่อง'
        datediff(to_date(current_timestamp()), col("timestamp_date"))
    )
)

# ตรวจสอบผลลัพธ์
df_final_ready.select(
    "ticket_id", "state", "lat", "lon", 
    "timestamp_dt", "last_activity_dt", "DaysToFix","timestamp_date", "last_activity_date"
).show(5, truncate=False)

+-----------+---------+--------+---------+-------------------+-------------------+---------+--------------+------------------+
|ticket_id  |state    |lat     |lon      |timestamp_dt       |last_activity_dt   |DaysToFix|timestamp_date|last_activity_date|
+-----------+---------+--------+---------+-------------------+-------------------+---------+--------------+------------------+
|2021-FYJTFP|เสร็จสิ้น|13.81865|100.53084|2021-09-03 12:51:09|2022-06-04 15:34:14|274      |2021-09-03    |2022-06-04        |
|2021-CGPMUN|เสร็จสิ้น|13.67891|100.66709|2021-09-19 14:56:08|2022-06-21 08:21:09|275      |2021-09-19    |2022-06-21        |
|2021-7XATFA|เสร็จสิ้น|13.7206 |100.52649|2021-09-26 05:03:52|2022-06-06 01:17:12|253      |2021-09-26    |2022-06-06        |
|2021-9U2NJT|เสร็จสิ้น|13.81853|100.53099|2021-10-14 10:45:27|2022-09-08 08:35:43|329      |2021-10-14    |2022-09-08        |
|2021-DVEWYM|เสร็จสิ้น|13.8228 |100.59165|2021-12-09 12:29:08|2022-08-12 07:18:44|246      |2021-12-09    |2022

In [14]:

df_clean = (
    df_final_ready
    .withColumn("comment_clean", trim(col("comment")))
    .withColumn("comment_clean", lower(col("comment_clean")))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[\n\r\t]", " "))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[^ก-๙a-z0-9/. ]", ""))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), " +", " "))
)

MIN_COMMENT_LENGTH = 10
df_clean = df_clean.filter(
    (length(col("comment")) >= MIN_COMMENT_LENGTH)
)
df_clean.select("comment_clean").show(20, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|comment_clean                                                                                                                                                                                                                                                                       |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|น้ำท่วมเวลาฝนตกและทะลุเข้าบ้านเดือดร้อนมากทุกๆปีจะมีเครื่องสูบน้ำแต่ปีนี้ไม่มีกทม.ปล่อยทิ้ง ชุมชนเคหะนคร1แปลง2ซ.เฉลิมพระเกียรติร.9ซอง22 วัดตะกล่ำ ประเวศ          

In [15]:
# df_clean = df_clean.filter(col("comment_clean").isNotNull())
# print(df_clean.count())

In [16]:

# @udf(ArrayType(StringType()))
# def thai_tokenize(text):
#     if text and text.strip():
#         # ใช้ Engine 'newmm' และลบช่องว่างออกจาก tokens
#         return word_tokenize(text, engine='newmm', keep_whitespace=False)
#     else:
#         return []

# # UDF สำหรับการลบ Stopwords
# # โหลด Stopwords List แค่ครั้งเดียว
# STOPWORDS = set(thai_stopwords())
# @udf(ArrayType(StringType()))
# def remove_stopwords(tokens):
#     if tokens is not None:
#         return [word for word in tokens if word not in STOPWORDS and word != '']
#     else:
#         return []


# # A. สร้างคอลัมน์ tokens จาก 'comment_clean'
# df_tokenized = df_clean.withColumn(
#     "tokens", 
#     thai_tokenize(col("comment_clean")) # แก้ไขเป็น 'comment_clean' แล้ว
# )

# # B. สร้างคอลัมน์ final_tokens (สำหรับ LLM Sentiment)
# df_final_llm = df_tokenized.withColumn(
#     "final_tokens", 
#     remove_stopwords(col("tokens"))
# )

# df_final_llm.select("comment_clean", "final_tokens").show(5, truncate=False)

In [17]:

# df_clean.select([
#     sum(col(c).isNull().cast("int")).alias(c)
#     for c in df_clean.columns
# ]).show()


In [18]:
# df_clean.describe().show()

In [19]:
# คอลัมน์ที่เราต้องการเก็บไว้เท่านั้น
COLUMNS_TO_KEEP = [
    "ticket_id",
    "type",
    # "organization",
    "state",
    "address",
    "district",
    "province",
    "subdistrict",
    
    # Core features
    "lat",
    "lon",
    "DaysToFix",
    
    # Text Input for LLM
    "comment_clean",
    
    # Time Analysis (DateTimes)
    "timestamp_dt",
    "last_activity_dt"
]

df_ready_for_export = df_clean.select(*COLUMNS_TO_KEEP)

print(f"จำนวนคอลัมน์เดิม: {len(df_final_ready.columns)}")
print(f"จำนวนคอลัมน์ใหม่: {len(df_ready_for_export.columns)}")

df_ready_for_export.printSchema()

จำนวนคอลัมน์เดิม: 27
จำนวนคอลัมน์ใหม่: 13
root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- state: string (nullable = true)
 |-- address: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- DaysToFix: integer (nullable = true)
 |-- comment_clean: string (nullable = true)
 |-- timestamp_dt: timestamp (nullable = true)
 |-- last_activity_dt: timestamp (nullable = true)



In [20]:
from pyspark.sql.functions import lit


df_model_ready = df_ready_for_export.withColumn(
    "is_fixed",
    when(col("state") == "เสร็จสิ้น", lit(1)).otherwise(lit(0))
)

# ตอนนี้ DaysToFix และ is_fixed สามารถใช้เป็น Features ในโมเดลได้
# df_model_ready.select("DaysToFix", "is_fixed","state", "comment_clean").show(10)

df_multi_hot = df_model_ready

for t in livability_types:
    # สร้างชื่อคอลัมน์ใหม่ (เช่น type_ถนน)
    new_col_name = f"type_{t}"
    
    df_multi_hot = df_multi_hot.withColumn(
        new_col_name,
        # ถ้าคอลัมน์ 'type' ดั้งเดิม มีข้อความ 't' อยู่ ให้กำหนดค่าเป็น 1
        when(col("type").contains(t), lit(1)).otherwise(lit(0))
    )

# --- ตรวจสอบผลลัพธ์ ---
# เลือกคอลัมน์ type ดั้งเดิม และคอลัมน์ Multi-Hot ที่สร้างขึ้นใหม่
selected_cols = ["ticket_id", "type"] + [f"type_{t}" for t in livability_types]
selected_cols.pop(-3)
selected_cols.append("type_PM25")
df_multi_hot = df_multi_hot.withColumnRenamed("type_PM2", "type_PM25")

df_multi_hot.select(*selected_cols).show(20, truncate=False)

+-----------+---------------------+--------+------------+----------------+-------------+--------------+------------+----------------+------------+-----------+----------+----------+---------+
|ticket_id  |type                 |type_ถนน|type_ทางเท้า|type_ความปลอดภัย|type_แสงสว่าง|type_ความสะอาด|type_กีดขวาง|type_ท่อระบายน้ำ|type_น้ำท่วม|type_ต้นไม้|type_จราจร|type_สะพาน|type_PM25|
+-----------+---------------------+--------+------------+----------------+-------------+--------------+------------+----------------+------------+-----------+----------+----------+---------+
|2021-CGPMUN|{น้ำท่วม,ร้องเรียน}  |0       |0           |0               |0            |0             |0           |0               |1           |0          |0         |0         |0        |
|2021-7XATFA|{สะพาน}              |0       |0           |0               |0            |0             |0           |0               |0           |0          |0         |1         |0        |
|2021-DVEWYM|{น้ำท่วม,ถนน}        |1       |0

In [ ]:
df_final_ml = df_multi_hot.drop("type", "state","organization") 
# ถ้าคุณสร้างคอลัมน์ 'type_clean' ชั่วคราวในการแก้ไขปัญหา ก็ควรลบคอลัมน์นั้นด้วย
# df_final_ml = df_multi_hot.drop("type", "state", "type_clean") 

print(f"จำนวนคอลัมน์หลัง Drop: {len(df_final_ml.columns)}")
print("ตัวอย่าง Schema หลัง Drop:")
df_final_ml.printSchema()

จำนวนคอลัมน์หลัง Drop: 24
ตัวอย่าง Schema หลัง Drop:
root
 |-- ticket_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- DaysToFix: integer (nullable = true)
 |-- comment_clean: string (nullable = true)
 |-- timestamp_dt: timestamp (nullable = true)
 |-- last_activity_dt: timestamp (nullable = true)
 |-- is_fixed: integer (nullable = false)
 |-- type_ถนน: integer (nullable = false)
 |-- type_ทางเท้า: integer (nullable = false)
 |-- type_ความปลอดภัย: integer (nullable = false)
 |-- type_แสงสว่าง: integer (nullable = false)
 |-- type_ความสะอาด: integer (nullable = false)
 |-- type_กีดขวาง: integer (nullable = false)
 |-- type_ท่อระบายน้ำ: integer (nullable = false)
 |-- type_น้ำท่วม: integer (nullable = false)
 |-- type_ต้นไม้: integer (nullable = false)
 |-- type_PM25: integer 

In [ ]:
# อันนี้เซฟเเยกไฟล์
# from pyspark.sql import functions as F
# import pandas as pd
# import math
# import os
# import shutil
# OUTPUT_DIR = "C:/temp/spark_batch_output"
# shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # เพิ่ม column row_id เพื่อแบ่ง batch
# df_indexed = df_final_ml.withColumn("row_id", F.monotonically_increasing_id())

# batch_size = 100000
# total_rows = 596369  # กำหนดตรง เพราะ count() crash
# num_batches = math.ceil(total_rows / batch_size)

# for i in range(num_batches):
#     start = i * batch_size
#     end = start + batch_size
#     df_batch = df_indexed.filter((F.col("row_id") >= start) & (F.col("row_id") < end)).drop("row_id")
#     pdf = df_batch.toPandas()
#     pdf.to_csv(f"{OUTPUT_DIR}/spark_batch_{i+1}.csv", index=False)


#------------------------------------------------
# อันนี้โค้ดเซฟที่ไม่ได้ ลองเซฟแค่10แถวแรกดู
df_test_tiny_subset = df_final_ml.limit(10)
OUTPUT_PATH_SUBSET = "C:/temp/spark_test_subset_utf8"

df_test_tiny_subset.write.format("csv") \
                          .option("header", "true") \
                          .mode("overwrite") \
                          .save(OUTPUT_PATH_SUBSET)
print("Subset test save complete with UTF-8 encoding.")

In [ ]:
df_map_sample_pandas = df_ready_for_export.sample(
    fraction=10000 / df_ready_for_export.count(), 
    seed=42
).toPandas()
display(df_map_sample_pandas.head(10))

,ticket_id,type,state,address,district,province,subdistrict,lat,lon,DaysToFix,comment_clean,timestamp_dt,last_activity_dt
0,2022-E449PD,{ถนน},เสร็จสิ้น,13/131 ถนน ศรีนครินทร์ แขวง หนองบอน เขต ประเวศ...,ประเวศ,กรุงเทพมหานคร,หนองบอน,13.69141,100.647301,144,เส้นทางม้าลายไม่ชัด ตรงหน้าทางเข้าตลาดนัดรถไฟ ...,2022-01-30 15:43:46,2022-06-23 23:48:29
1,2022-GA7ANZ,{ถนน},เสร็จสิ้น,324/326 ถนน ประชาราษฎร์ สาย 1 แขวง บางซื่อ เขต...,บางซื่อ,กรุงเทพมหานคร,บางซื่อ,13.80746,100.521637,60,ใช่ค่ะ เมื่อวานโดนกับตัวเลยค่ะ รถเกือบล้ม ที่ถ...,2022-05-27 10:13:32,2022-07-26 04:40:16
2,2022-4Y426T,"{น้ำท่วม,ถนน,ความปลอดภัย}",เสร็จสิ้น,460 ซอยลาดกระบัง 52 แยก 2-2-1 แขวง ลาดกระบัง เ...,ลาดกระบัง,กรุงเทพมหานคร,ลาดกระบัง,13.72049,100.782433,149,1. ซอยลาดกระบัง 50/2 ถนนพุพัง และน้ำท่วมบ่อยคร...,2022-05-29 10:16:48,2022-10-25 07:57:50
3,2022-NLLWWH,{ทางเท้า},เสร็จสิ้น,276 ซอย สุขุมวิท 93 แขวง บางจาก เขตพระโขนง กรุ...,พระโขนง,กรุงเทพมหานคร,บางจาก,13.70028,100.608643,164,ประเภททางเท้า,2022-05-29 10:24:58,2022-11-09 06:25:25
4,2022-K6D7AF,{ท่อระบายน้ำ},เสร็จสิ้น,6/2 ถนน เคหะร่มเกล้า แขวง คลองสองต้นนุ่น เขตลา...,ลาดกระบัง,กรุงเทพมหานคร,คลองสองต้นนุ่น,13.76791,100.719070,149,ปัญหามีหลุมที่เสี่ยงต่อคนเดินตกท่อไม่มีฝาปิดที...,2022-05-29 11:07:34,2022-10-25 07:56:59
5,2022-EF7EYG,"{ต้นไม้,ถนน,แสงสว่าง}",เสร็จสิ้น,Rca Opposite Kromadit Building แขวง บางกะปิ เข...,ห้วยขวาง,กรุงเทพมหานคร,บางกะปิ,13.74612,100.579071,813,ป้ายรถเมล์อาร์ซีเอตรงหน้าถนนเพชรบุรีตัดใหม่เป็...,2022-05-29 11:17:40,2024-08-19 02:34:02
6,2022-NRGHK9,{กีดขวาง},เสร็จสิ้น,477 21 ถ. จรัญสนิทวงศ์ แขวง บางขุนศรี เขตบางกอ...,บางกอกน้อย,กรุงเทพมหานคร,บางขุนศรี,13.75943,100.469612,13,มอไซจอดรถบนถนนขวางทางเดินรถ ขับรถสวนกันไม่ได้,2022-05-29 12:07:04,2022-06-11 11:27:36
7,2022-GNFNEV,{ถนน},เสร็จสิ้น,150/411 ถ. พระรามที่ 2 แขวง ท่าข้าม เขตบางขุนเ...,บางขุนเทียน,กรุงเทพมหานคร,ท่าข้าม,13.66868,100.449951,19,1.ถนนเส้นพระรามสอง ทราบว่ากำลังก่อสร้างทางด่วน...,2022-05-29 13:52:18,2022-06-17 10:48:52
8,2022-9Z7TA6,{ท่อระบายน้ำ},เสร็จสิ้น,812 23 ปาก ซอย ประชาชื่น 24 แขวง วงศ์สว่าง เขต...,บางซื่อ,กรุงเทพมหานคร,วงศ์สว่าง,13.82364,100.537277,119,ซอยประชาชื่น24เข้ามา200เมตร ท่อตันน้ำไม่ระบาย ...,2022-05-30 01:45:35,2022-09-26 09:12:40
9,2022-8XDNXD,{ถนน},เสร็จสิ้น,409 ถ. สุคนธสวัสดิ์ แขวงลาดพร้าว เขตลาดพร้าว ก...,ลาดพร้าว,กรุงเทพมหานคร,ลาดพร้าว,13.83187,100.624023,122,มีเรื่องแจ้งเพิ่มเติมอีก 1 เรื่องครับ บรเวณถนน...,2022-05-30 04:48:30,2022-09-29 03:32:17


In [ ]:
# # -----------------------------
# # 1️⃣ Import libraries
# # -----------------------------
# from pyspark.sql.functions import col, split
# import pandas as pd
# import numpy as np
# import folium
# from folium.plugins import HeatMap

# # -----------------------------
# # 2️⃣ แยก lon / lat จาก coords string
# # -----------------------------
# # สมมติ coords เป็น "lon,lat"
# df_final_ready = df_final_ready.withColumn("lon", split(col("coords"), ",").getItem(0).cast("double")) \
#                                .withColumn("lat", split(col("coords"), ",").getItem(1).cast("double"))

# # -----------------------------
# # 3️⃣ Sample data (10,000 rows) และแปลงเป็น Pandas
# # -----------------------------
# n_sample = 10000
# n_total = df_final_ready.count()
# df_sample_pandas = df_final_ready.sample(fraction=n_sample / n_total, seed=42).toPandas()

# # -----------------------------
# # 4️⃣ เตรียม weight (DaysToFix) แบบ log scale
# # -----------------------------
# df_sample_pandas['weight'] = np.log1p(df_sample_pandas['DaysToFix'])

# # -----------------------------
# # 5️⃣ เตรียมข้อมูลสำหรับ HeatMap
# # -----------------------------
# data_heatmap = df_sample_pandas[['lat', 'lon', 'weight']].values.tolist()

# # -----------------------------
# # 6️⃣ สร้าง Folium Map
# # -----------------------------
# center_lat = 13.737
# center_lon = 100.528

# m = folium.Map(
#     location=[center_lat, center_lon],
#     zoom_start=11,
#     tiles="cartodbpositron"
# )

# # -----------------------------
# # 7️⃣ เพิ่ม HeatMap Layer
# # -----------------------------
# HeatMap(
#     data_heatmap,
#     radius=10,            # ขนาดจุด
#     blur=15,              # ความฟุ้ง
#     max_val=df_sample_pandas['weight'].max()
# ).add_to(m)

# # -----------------------------
# # 8️⃣ แสดงผล (Jupyter Notebook) / บันทึกเป็น HTML
# # -----------------------------
# m  # Interactive map in notebook
# m.save("daystofix_heatmap.html")  # บันทึกเป็น HTML
